In [34]:
from moviepy import VideoFileClip

video = VideoFileClip("audio/meeting.mp4")
video.audio.write_audiofile("audio/meeting.wav")

MoviePy - Writing audio in audio/meeting.wav


MoviePy - Done.


In [35]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from pyannote.audio import Pipeline
from pathlib import Path


In [43]:
# =========================
# AYARLAR
# =========================
AUDIO_FILE = "./audio/meeting.wav"
WHISPER_DIR = "./whisper-medium"
DIARIZATION_CONFIG = Path("./speaker-diarization-3.1/config.yaml")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_ID = 0 if torch.cuda.is_available() else -1
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

MIN_SPEAKERS = 2      # bilmiyorsan None yap
MAX_SPEAKERS = 2      # bilmiyorsan None yap
MERGE_GAP_SECONDS = 1.0

In [37]:
# =========================
# 1) WHISPER
# =========================
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_DIR,
    local_files_only=True,
    torch_dtype=TORCH_DTYPE,
)

processor = AutoProcessor.from_pretrained(
    WHISPER_DIR,
    local_files_only=True,
)

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=TORCH_DTYPE,
    device=DEVICE_ID,
)

asr_result = asr_pipe(
    AUDIO_FILE,
    return_timestamps=True,
    generate_kwargs={
        "language": "tr",
        "task": "transcribe",
        "temperature": 0.0,
        "no_repeat_ngram_size": 3,
    },
)

asr_chunks = []
for ch in asr_result.get("chunks", []):
    ts = ch.get("timestamp")
    if not ts:
        continue

    start, end = ts
    if start is None or end is None:
        continue

    text = ch.get("text", "").strip()
    if not text:
        continue

    asr_chunks.append({
        "start": float(start),
        "end": float(end),
        "text": text,
    })

Device set to use cpu
/home/cypher/.local/lib/python3.10/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


In [44]:
# =========================
# 2) PYANNOTE
# =========================
diar_pipeline = Pipeline.from_pretrained(DIARIZATION_CONFIG)

if DEVICE == "cuda":
    diar_pipeline.to(torch.device("cuda"))

diar_kwargs = {}
if MIN_SPEAKERS is not None:
    diar_kwargs["min_speakers"] = MIN_SPEAKERS
if MAX_SPEAKERS is not None:
    diar_kwargs["max_speakers"] = MAX_SPEAKERS

diarization = diar_pipeline(AUDIO_FILE, **diar_kwargs)

speaker_turns = []
for turn, _, speaker in diarization.itertracks(yield_label=True):
    speaker_turns.append({
        "start": float(turn.start),
        "end": float(turn.end),
        "speaker": speaker,
    })

/home/cypher/.local/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
/home/cypher/.local/lib/python3.10/site-packages/pya

In [45]:
# =========================
# 3) EŞLEŞTİRME
# =========================
def overlap(a_start, a_end, b_start, b_end):
    return max(0.0, min(a_end, b_end) - max(a_start, b_start))

mapped_chunks = []
for chunk in asr_chunks:
    best_speaker = "UNKNOWN"
    best_overlap = -1.0

    for spk in speaker_turns:
        ov = overlap(chunk["start"], chunk["end"], spk["start"], spk["end"])
        if ov > best_overlap:
            best_overlap = ov
            best_speaker = spk["speaker"]

    mapped_chunks.append({
        "start": chunk["start"],
        "end": chunk["end"],
        "speaker": best_speaker,
        "text": chunk["text"],
    })

In [46]:
# =========================
# 4) Speaker 1 / Speaker 2
# =========================
speaker_map = {}
speaker_idx = 1

for item in mapped_chunks:
    spk = item["speaker"]
    if spk not in speaker_map:
        speaker_map[spk] = f"Speaker {speaker_idx}"
        speaker_idx += 1
    item["speaker"] = speaker_map[spk]

# =========================
# 5) AYNI KONUŞMACIYI BİRLEŞTİR
# =========================
dialogues = []
for item in mapped_chunks:
    if not dialogues:
        dialogues.append(item.copy())
        continue

    prev = dialogues[-1]
    same_speaker = prev["speaker"] == item["speaker"]
    close_in_time = (item["start"] - prev["end"]) < MERGE_GAP_SECONDS

    if same_speaker and close_in_time:
        prev["text"] += " " + item["text"]
        prev["end"] = item["end"]
    else:
        dialogues.append(item.copy())


In [47]:
# =========================
# 5) AYNI KONUŞMACIYI BİRLEŞTİR
# =========================
dialogues = []
for item in mapped_chunks:
    if not dialogues:
        dialogues.append(item.copy())
        continue

    prev = dialogues[-1]
    same_speaker = prev["speaker"] == item["speaker"]
    close_in_time = (item["start"] - prev["end"]) < MERGE_GAP_SECONDS

    if same_speaker and close_in_time:
        prev["text"] += " " + item["text"]
        prev["end"] = item["end"]
    else:
        dialogues.append(item.copy())

In [48]:
# =========================
# 6) ÇIKTI
# =========================
for d in dialogues:
    print(f'{d["speaker"]}: {d["text"]}')

with open("dialog_output.txt", "w", encoding="utf-8") as f:
    for d in dialogues:
        f.write(f'{d["speaker"]}: {d["text"]}\n')

Speaker 1: Yelizhan Çalık, 25 Haziran 2004. Donluyu. Bu senin listeden mezun oldum. Anne adım Öznür, baba adım Hasan. Annem güzellik uzmanı. Babam ise müteahhitlik mesleğiyle uğraşıyor. Müteahitlik? Evet. Anladım. İki kardeşiz, en büyüğü benim.
Speaker 2: Neden polis olmak istiyorsun peki?
Speaker 1: Çok sevdiğim saygıdeğer bir eniştem vardı. Kendisi bomba uzman, aynı zamanda da emniyet amiriydi.
Speaker 2: Bomba imhamı o zaman?
Speaker 1: Aynen, bomba imha uzmanıydı. Tam emniyet müdürü olmasına birkaç gün kala şehit oldu. Ben de onun yolundan gitmek istiyorum Allah nasip ederse. Bir tane soru çekebilir misin? Tabii. Parakter nedir? Parakter kişinin kendini yaşadığı çevrenin de etkisiyle yaşadığın psikolojik veya bir yer olaylardan ötürü dış görüşüne yansımasıdır. Kötü görmedim.
Speaker 1: Verdiğin örnek çok iyi. Neden polis olmak istiyorsun sorusuna cevabın iyiydi.
Speaker 2: Beden dilin iyi.
Speaker 1: Ses tonun güzel. Dokusana yakın puan veriyorum. Hatta doksan veriyor.
